# INF6083 - Projet P2
## Recommandation basée sur le contenu - Amazon Reviews 2023 (Books)
### Équipe 7

**Contexte :** Filtrage basé sur le contenu à partir des métadonnées des livres. Fichier central : `meta_Books.jsonl.gz` (parent_asin, title, description, categories, average_rating, rating_number, price). Même sous-ensemble de travail que le projet P1.

## 0.1 Importation des bibliothèques

In [3]:
# -- Bibliothèques standard --------------------------------------
import glob
import os
import gc
from pathlib import Path

# -- Calcul scientifique ------------------------------------------
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

# -- Scikit-learn -------------------------------------------------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

from scripts.precursor import PROCESSED_DATA_DIR

# -- Optionnel : PyTorch (GPU/CPU) pour Tâche 3 --------------------
# import torch


# -- Verification Précurseur ---------------------------------------
from main import main
main()



 final_files_checker : True


 Echantillon present 

Reutilisation des ensembles P1
Jointure chemin : data/joining/active_post_split_union_joined.parquet
Jointure chemin : data/joining/active_pre_split_joined.parquet
Jointure chemin : data/joining/temporal_post_split_union_joined.parquet
Jointure chemin : data/joining/temporal_pre_split_joined.parquet
# Diagnostic Task 0 — Préparation des données (P2)

- generated_at: 2026-03-16T21:13:23

## A. Réutilisation du sous-ensemble de travail
- note: `P2 réutilise les sous-ensembles P1 (active/temporal, filtered + splits).`
- methodological_note: `Aucun nouvel échantillonnage massif du corpus complet n'est effectué; les jeux issus de P1 sont réexploités pour la fusion avec meta_Books.`

## B. Documentation des sources

### active_pre_split
- stage: `pre_split`
- variant: `active`
- role: `interactions`
- kind: `single`
- exists: `True`
- format: `parquet`
- rows: `508878`
- cols: `10`
- size_bytes: `318642272`
- paths: `['data/processed/samp

## 0.2 Chemins et chargement des données
Configuration des chemins vers les métadonnées et les interactions (sous-ensemble P1).

In [ ]:
# Chargement métadonnées (meta_Books.jsonl.gz) et interactions P1
# À adapter selon l'emplacement réel des fichiers produits en P1

RAW_DATA_DIR = Path("data/raw/parquet/")
PROCESSED_DATA_DIR = Path("data/processed/")

META_PARQUET_PATH = RAW_DATA_DIR / "meta_Books.parquet"

SAMPLE_ACTIVE_DIR = f"{PROCESSED_DATA_DIR}sample-active-users"
SAMPLE_TEMPORAL_DIR = f"{PROCESSED_DATA_DIR}sample-temporal"

SAMPLE_GLOB_ORIGINAL = f"{PROCESSED_DATA_DIR}sample-*/*_original.parquet"
SAMPLE_GLOB_CLEANED = f"{PROCESSED_DATA_DIR}sample-*/*_cleaned.parquet"
SAMPLE_GLOB_FILTERED = f"{PROCESSED_DATA_DIR}sample-*/*_filtered.parquet"
SAMPLE_GLOB_FILTERED_LIST = sorted(glob.glob(SAMPLE_GLOB_FILTERED))

# 3.1 Tâche 0 - Préparation des données et construction du cadre expérimental

## 3.1.1 Réutilisation du sous-ensemble de travail

- Vérifier la qualité de la jointure entre les interactions et les métadonnées via `parent_asin`.
- Identifier les attributs effectivement exploitables.
- Documenter les valeurs manquantes et les choix de traitement retenus.
- Produire un jeu de données final cohérent pour le reste du projet.

## 3.1.2 Division entraînement/test

- Division temporelle par utilisateur : les interactions les plus récentes de chaque utilisateur sont réservées au test, le reste sert à l'entraînement.
- Garantir que chaque utilisateur présent dans le test dispose d'un historique exploitable dans l'entraînement.
- S'assurer qu'aucune information du test n'est utilisée lors de la construction des représentations ou du réglage des méthodes.

In [ ]:
# Split temporel par utilisateur (train = anciennes, test = plus récentes)
pass

## 3.1.3 Construction des représentations d'items

### 1) Représentation textuelle
- Récupération des données

In [5]:
books_data = pd.read_parquet("data/joining/active_post_split_union_joined.parquet")

- Utilisation des données textuelles pertinentes : titre, description et catégorie(s) du livre

In [ ]:
# Formattage éventuel des catégories
if isinstance(books_data.at[0, "categories"], str):
    books_data["categories"] = books_data["categories"].apply(lambda x: x.split(", "))

# Combinaison des colonnes 'title', 'description' et 'categories'
books_data["combined_infos"] = books_data.apply(
    lambda row: f"{row['title']}. {row['description']}. {' '.join(row['categories'])}",
    axis=1
)

# Vérification
print(books_data["combined_infos"].iloc[0])


Bones of the Lost: A Temperance Brennan Novel (16). From Publishers Weekly | Bestseller Reichs draws on her experiences touring with the USO in Afghanistan for her captivating 16th novel featuring forensic anthropologist Temperance Brennan (after 2012's Bones Are Forever). At home in Charlotte, N.C., the bone expert concludes that the death of an unidentified girl, 14 or 15 years old, was caused by foul play rather than a hit-and-run, as was previously suspected. The outraged Brennan urges homicide detective Erskine Skinny Slidell to investigate, knowing Slidell believes the girl to have been an undocumented immigrant, as well as possibly being a junkie and prostitute. Later in Afghanistan, Brennan oversees the exhumation of two unarmed Afghan villagers killed by a U.S. Marine to determine whether the victims were shot in the back or head-on. The two cases—and a third involving mummified dogs from Peru—give Reichs ample opportunity to provide detailed descriptions of forensic examinati

- Prétraitement des textes

In [16]:
import re
from unidecode import unidecode

# Gestion des stop words
STOPWORDS = {
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours",
    "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers",
    "herself", "it", "its", "itself", "they", "them", "their", "theirs", "themselves",
    "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are",
    "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", "does",
    "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because", "as", "until",
    "while", "of", "at", "by", "for", "with", "about", "against", "between", "into",
    "through", "during", "before", "after", "above", "below", "to", "from", "up", "down",
    "in", "out", "on", "off", "over", "under", "again", "further", "then", "once", "here",
    "there", "when", "where", "why", "how", "all", "any", "both", "each", "few", "more",
    "most", "other", "some", "such", "no", "nor", "not", "only", "own", "same", "so", "than",
    "too", "very", "s", "t", "can", "will", "just", "don", "should", "now"
}

# Gestion de la ponctuation
PUNCTUATION_TABLE = str.maketrans("", "", "!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~")

def clean_text(text):
    # Suppression des URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Suppression de la ponctuation
    text = text.translate(PUNCTUATION_TABLE)
    # Suppression des chiffres
    text = re.sub(r'\d+', '', text)
    # Conversion en minuscules et enlèvement des accents
    text = unidecode(text.lower())
    # Tokenisation et filtrage des stopwords
    tokens = text.split()
    tokens = [
        token for token in tokens 
        if token not in STOPWORDS 
        and len(token) > 2
        and token.isalpha()
        and not re.search(r'(.)\1{2,}', token)
    ]
    return " ".join(tokens)

batch_size = 10000
cleaned_texts = []

for i in range(0, len(books_data), batch_size):
    batch = books_data["combined_infos"].iloc[i:i+batch_size]
    cleaned_batch = batch.apply(clean_text)
    cleaned_texts.extend(cleaned_batch)

books_data["cleaned_infos"] = cleaned_texts

# Vérification
print(books_data["cleaned_infos"].iloc[0])

bones lost temperance brennan novel publishers weekly bestseller reichs draws experiences touring uso afghanistan captivating novel featuring forensic anthropologist temperance brennan bones forever home charlotte bone expert concludes death unidentified girl years old caused foul play rather hitandrun previously suspected outraged brennan urges homicide detective erskine skinny slidell investigate knowing slidell believes girl undocumented immigrant well possibly junkie prostitute later afghanistan brennan oversees exhumation two unarmed afghan villagers killed marine determine whether victims shot back headon two third involving mummified dogs reichs ample opportunity provide detailed descriptions forensic examinations brennans passionate personal involvement provides excitement masterful tale city author tour agent jennifer rudolph walsh william morris endeavor aug booklist usual forensic anthropologist temperance brennan juggling several cases including mummified dog remains could 

- Vectorisation (TF-IDF)

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorisation TF-IDF
vectorizer = TfidfVectorizer(min_df=5, max_df=0.95, ngram_range=(1, 1))
X_tfidf = vectorizer.fit_transform(cleaned_texts)

# Vérifications
print(f"Dimension de la matrice TF-IDF : {X_tfidf.shape}")
print(f"Vocabulaire (10 premiers mots) : {vectorizer.get_feature_names_out()[:10]}")


Dimension de la matrice TF-IDF : (497931, 503758)
Vocabulaire (10 premiers mots) : ['aab' 'aabb' 'aabe' 'aabsolutely' 'aac' 'aacc' 'aachamberlynnfacebook'
 'aachen' 'aadam' 'aadding']


- Sur **497 931** livres, nous avons donc **503 758** mots uniques (n-grammes) d'après le filtrage donné. Nous avons des mots qui manquent de sens car nous avons privilégié des fonctions moins consommatrices en mémoire et performance pour notre taille d'échantillon ; des bibliothèques comme spaCy spécialisées dans le nettoyage des textes, et incluant notamment la lemmatisation (simplification des mots à leur sens de base), peuvent être intéressantes pour affiner ce travail.

### 2) Attributs structurés
- Intégration ou non d'attributs (average_rating, price, etc.) selon une stratégie argumentée.
### 3) Caractérisation
- Dimension, densité, qualité apparente et limites observables de la représentation obtenue.

## 3.1.4 Construction des profils utilisateurs

- À partir de l'ensemble d'entraînement, construire pour chaque utilisateur un profil contenu (agrégation des vecteurs des livres appréciés).
- Expliciter : quels items sont utilisés ; si les notes sont prises en compte comme pondération ; gestion des utilisateurs à historique très limité ; hypothèses implicites de la définition du profil.

In [ ]:
# Profil utilisateur = agrégation (moyenne pondérée ou non) des vecteurs items en train
pass

# 3.2 Tâche 1 - Recommandation par similarité de contenu

## 3.2.1 Principe

Pour chaque utilisateur cible, calculer un score de pertinence entre son profil et chaque livre non encore vu (entraînement). Une mesure de similarité adaptée (ex. cosinus pour TF-IDF) devra être choisie et justifiée.

## 3.2.2 Implémentation

- Mécanisme de recommandation top-N : pour chaque utilisateur de test, produire une liste ordonnée de livres recommandés.
- Implémentation efficace compte tenu de la dimension potentiellement élevée des représentations.

In [ ]:
# Similarité cosinus profil-item, top-N (exclure déjà vus), liste ordonnée par utilisateur de test
pass

## 3.2.3 Analyse qualitative

Pour quelques utilisateurs représentatifs, comparer l'historique de lecture avec les livres recommandés : cohérence thématique ; proximité apparente ; cas de recommandations plausibles mais redondantes ; situations où le modèle semble trop spécialisé.

In [ ]:
# Exemples utilisateurs : historique vs recommandations, commentaires qualitatifs
pass

# 3.3 Tâche 2 - Représentations latentes des contenus

## 3.3.1 Réduction de dimension

- Appliquer une méthode adaptée aux données textuelles (ex. SVD tronquée).
- Obtenir une représentation plus compacte tout en préservant une partie de l'information sémantique.
- Justifier le choix ; étudier plusieurs dimensions latentes ; analyser le compromis compacité / coût / qualité des recommandations.

In [ ]:
# SVD tronquée sur matrice items, étude de plusieurs dimensions latentes
pass

## 3.3.2 Projection des profils utilisateurs

Les profils utilisateurs doivent être exprimés dans le même espace latent que les items, afin de permettre la recommandation par similarité dans cet espace transformé.

In [ ]:
# Projection des profils dans l'espace latent, recommandation top-N dans cet espace
pass

## 3.3.3 Analyse

Comparer les recommandations avec celles de la Tâche 1 : qualité des listes ; effet de la dimension latente ; comportement selon la richesse des profils ; interprétabilité des dimensions latentes. Une visualisation 2D peut appuyer l'analyse.

In [ ]:
# Comparaison Tâche 1 vs Tâche 2, visualisation 2D optionnelle
pass

# 3.4 Tâche 3 - Apprentissage d'un score de pertinence

## 3.4.1 Formulation

Formulation au choix (à justifier) : régression (prédiction de note), classification binaire (pertinent ou non), ou apprentissage d'un score de classement.

## 3.4.2 Variables explicatives

Variables possibles : représentation de l'item ; profil utilisateur ; score de similarité profil-item ; caractéristiques globales du livre ou de l'utilisateur ; transformations ou interactions pertinentes.

## 3.4.3 Modèles

- Implémenter au moins deux modèles d'apprentissage distincts et comparer leur comportement.
- Protocole compatible avec le cadre expérimental (Tâche 0). Réglage des hyperparamètres expliqué de manière concise et rigoureuse.

In [ ]:
# Construction des variables (user, item, similarité, etc.), entraînement de 2 modèles, prédiction top-N
pass

## 3.4.4 Discussion

Montrer en quoi l'apprentissage permet, ou non, d'améliorer la recommandation par rapport aux approches directes. Discuter les coûts en complexité, interprétabilité et robustesse.

In [ ]:
# Tableaux ou commentaires pour la discussion (Tâche 3 vs Tâche 1/2)
pass

# 3.5 Tâche 4 - Évaluation des recommandations

## 3.5.1 Évaluation quantitative

- Protocole d'évaluation hors ligne cohérent ; métriques de classement top-N (Precision@k, Recall@k, NDCG@k, etc.).
- Comparer au minimum : approche par similarité directe ; approche en espace latent ; approche basée sur l'apprentissage.

In [ ]:
# Calcul Precision@k, Recall@k, NDCG@k pour les 3 approches ; tableaux comparatifs
pass

## 3.5.2 Évaluation par profils d'utilisateurs

Analyse différenciée selon le niveau d'activité : distinguer profils peu actifs, modérément actifs et très actifs.

In [ ]:
# Découpage par niveau d'activité, métriques par segment
pass

## 3.5.3 Diversité et surspécialisation

Ne pas se limiter à la pertinence : discuter la diversité intra-liste et la tendance à recommander des livres trop proches. Proposer une mesure simple de diversité et l'analyser.

In [ ]:
# Mesure de diversité intra-liste, analyse et discussion
pass

# 3.6 Tâche 5 - Discussion générale et ouverture

Synthèse critique : principaux enseignements ; limites des représentations de contenu ; cold start (utilisateurs et nouveaux livres) ; biais possibles ; complémentarité P1/P2 ; pistes vers un système hybride (signaux collaboratifs + contenu).

*(Rédaction détaillée dans le rapport PDF ; ce notebook peut contenir les tableaux et figures résumant les résultats.)*

In [ ]:
# Résumés numériques, tableaux ou figures pour le rapport (optionnel)
pass